<a href="https://colab.research.google.com/github/piasa19/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/piasa19/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [7]:
# Cell 1 — Connect to the FlyRank warehouse

!pip -q install huggingface_hub pandas pyarrow

from huggingface_hub import hf_hub_download
import pandas as pd
import getpass

# Enter your Hugging Face READ token
hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

# Download the sample parquet file
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=hf_token
)

# Load the data
df = pd.read_parquet(file_path)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Enter your Hugging Face READ token: ··········
Rows: 11694072
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Date range: 2026-06-01 to 2026-06-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
# Cell 2 — Fields: feature / label / context / excluded

# Feature fields:
# These are measurements available before the decision date.
feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_data_available",
    "gsc_data_available"
]

# Label:
# The future outcome we want to predict.
label_fields = [
    "future_30d_decline"
]

# Context fields:
# Used for identification, grouping, dates, or validation.
context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4"
]

# Excluded fields:
# These should not be used as model features because they can
# contain future information, identifiers, or leakage.
excluded_fields = [
    "future_window_metrics",
    "trend_direction",
    "trend_pct",
    "raw_url",
    "raw_query",
    "client_identifying_information"
]

print("FEATURES")
for x in feature_fields:
    if x in df.columns:
        print("✓", x)

print("\nLABEL")
for x in label_fields:
    print("•", x)

print("\nCONTEXT")
for x in context_fields:
    if x in df.columns:
        print("✓", x)

print("\nEXCLUDED")
for x in excluded_fields:
    print("•", x)

FEATURES
✓ gsc_impressions
✓ gsc_clicks
✓ gsc_sum_position
✓ gsc_avg_position
✓ ga4_pageviews
✓ ga4_data_available
✓ gsc_data_available

LABEL
• future_30d_decline

CONTEXT
✓ report_date
✓ client_hash_id
✓ content_hash_id
✓ client_has_gsc
✓ client_has_ga4

EXCLUDED
• future_window_metrics
• trend_direction
• trend_pct
• raw_url
• raw_query
• client_identifying_information


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# Cell 3 — Verify the data contract with queries

# 1. Basic counts and date window
print("=== 1. COUNTS AND DATE WINDOW ===")

print("Total rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content items:", df["content_hash_id"].nunique())
print("First date:", df["report_date"].min())
print("Last date:", df["report_date"].max())


# 2. Check the declared grain
# One row should represent:
# report_date + client_hash_id + content_hash_id

print("\n=== 2. GRAIN CHECK ===")

grain_counts = (
    df.groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
)

duplicate_grain_rows = (grain_counts > 1).sum()

print("Number of duplicate grain combinations:", duplicate_grain_rows)

if duplicate_grain_rows == 0:
    print("PASS: One row = one report date + client + content item")
else:
    print("CHECK: Duplicate grain combinations exist")


# 3. Missing values
print("\n=== 3. MISSING VALUES ===")

important_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "gsc_data_available",
    "ga4_data_available"
]

missing_table = df[important_columns].isna().sum().sort_values(ascending=False)

print(missing_table)


# 4. Data availability flags
print("\n=== 4. DATA AVAILABILITY ===")

print("GSC availability:")
print(df["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 availability:")
print(df["ga4_data_available"].value_counts(dropna=False))


# 5. Check the 90-day feature / 30-day target window rule
print("\n=== 5. WINDOW CHECK ===")

first_date = pd.to_datetime(df["report_date"]).min()
last_date = pd.to_datetime(df["report_date"]).max()

earliest_decision_date = first_date + pd.Timedelta(days=90)
latest_decision_date = last_date - pd.Timedelta(days=30)

print("Data starts:", first_date.date())
print("Data ends:", last_date.date())
print("Earliest eligible decision date:", earliest_decision_date.date())
print("Latest eligible decision date:", latest_decision_date.date())

print(
    "\nA 90-day historical feature window and a following "
    "30-day target window do not overlap."
)

=== 1. COUNTS AND DATE WINDOW ===
Total rows: 11694072
Unique clients: 65
Unique content items: 409205
First date: 2026-06-01
Last date: 2026-06-30

=== 2. GRAIN CHECK ===
Number of duplicate grain combinations: 6390
CHECK: Duplicate grain combinations exist

=== 3. MISSING VALUES ===
gsc_avg_position      7815170
ga4_data_available    2397428
ga4_pageviews         2397428
report_date                 0
client_hash_id              0
gsc_clicks                  0
gsc_impressions             0
content_hash_id             0
gsc_data_available          0
dtype: int64

=== 4. DATA AVAILABILITY ===
GSC availability:
gsc_data_available
False    7815135
True     3878937
Name: count, dtype: int64

GA4 availability:
ga4_data_available
False    8651918
None     2397428
True      644726
Name: count, dtype: int64

=== 5. WINDOW CHECK ===
Data starts: 2026-06-01
Data ends: 2026-06-30
Earliest eligible decision date: 2026-08-30
Latest eligible decision date: 2026-05-31

A 90-day historical feature win

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
# Cell 4 — Data limits and limitations

print("=== DATA LIMITS ===")

print("""
1. The current sample contains only the latest full month:
   2026-06-01 to 2026-06-30.

2. Therefore, this sample alone cannot provide a complete
   90-day historical feature window plus a following 30-day
   target window.

3. Some fields contain missing values. In particular,
   GSC average position and GA4 measurements are not available
   for every row.

4. GA4 availability is uneven across the data. A missing or
   unavailable GA4 measurement should not automatically be
   interpreted as zero engagement.

5. The sample contains repeated combinations of
   report_date + client_hash_id + content_hash_id.
   Therefore, the declared grain must be checked against the
   full warehouse schema before treating this combination as
   unique.

6. Client histories are unbalanced, so different clients may
   have different amounts of historical data.

7. The data is observational. It can identify measured or
   directional signals associated with content performance,
   but it cannot prove that refreshing a page will cause
   performance to improve.

8. Future-window information must remain excluded from model
   features to prevent data leakage.
""")

# Inspect the duplicate grain combinations found in the sample
print("\n=== DUPLICATE GRAIN INSPECTION ===")

duplicate_rows = (
    df.groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
    .reset_index(name="row_count")
)

duplicate_rows = duplicate_rows[
    duplicate_rows["row_count"] > 1
].sort_values("row_count", ascending=False)

print("Number of duplicated grain combinations:",
      len(duplicate_rows))

print("\nTop duplicate combinations:")
display(duplicate_rows.head(10))

=== DATA LIMITS ===

1. The current sample contains only the latest full month:
   2026-06-01 to 2026-06-30.

2. Therefore, this sample alone cannot provide a complete
   90-day historical feature window plus a following 30-day
   target window.

3. Some fields contain missing values. In particular,
   GSC average position and GA4 measurements are not available
   for every row.

4. GA4 availability is uneven across the data. A missing or
   unavailable GA4 measurement should not automatically be
   interpreted as zero engagement.

5. The sample contains repeated combinations of
   report_date + client_hash_id + content_hash_id.
   Therefore, the declared grain must be checked against the
   full warehouse schema before treating this combination as
   unique.

6. Client histories are unbalanced, so different clients may
   have different amounts of historical data.

7. The data is observational. It can identify measured or
   directional signals associated with content performance,
   

,report_date,client_hash_id,content_hash_id,row_count
11659879,2026-06-30,client_e00b29e582949543,content_ff881e76e85001f4,2
4737329,2026-06-13,client_06d356715a8ff3b6,content_01dd034a486dd411,2
4737341,2026-06-13,client_06d356715a8ff3b6,content_03a8b5950a52518a,2
4737511,2026-06-13,client_06d356715a8ff3b6,content_1fa28219b658d722,2
4737559,2026-06-13,client_06d356715a8ff3b6,content_27a329abc15773a7,2
4737579,2026-06-13,client_06d356715a8ff3b6,content_2b4f82281f289e87,2
11655641,2026-06-30,client_def0955f7a377868,content_e6ef0149190d6370,2
11655432,2026-06-30,client_def0955f7a377868,content_c8dcbbd5d5fcf2d2,2
11655404,2026-06-30,client_def0955f7a377868,content_c52c3798a1412de7,2
11655226,2026-06-30,client_def0955f7a377868,content_ac604330c75afd04,2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.